# NGIML Inference on `juhenes/ngiml-test`

Downloads prepared test data from Hugging Face, runs inference, writes CSV outputs to Google Drive, and saves first 5 visualizations per dataset.

In [ ]:
from __future__ import annotations

import io
import json
import tarfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

CHECKPOINT_PATH = Path('/content/drive/MyDrive/ngiml_checkpoints/best_checkpoint.pt')
HF_DATASET_REPO_ID = 'juhenes/ngiml-test'
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/ngiml_test_inference')
HF_SNAPSHOT_LOCAL_DIR = Path('/content/hf_datasets/ngiml_test')

INFERENCE_STRATEGY = 'direct'
THRESHOLD_FOR_METRICS = None
PLOT_BINARY_THRESHOLD = 0.5

CSV_OUTPUT_DIR = DRIVE_OUTPUT_ROOT / 'csv'
PLOT_OUTPUT_DIR = DRIVE_OUTPUT_ROOT / 'plots'
CSV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents, Path('/content/ngiml')]:
        if (p / 'tools' / 'infer_helpers.py').exists() and (p / 'src').exists():
            return p
    raise FileNotFoundError('Could not find NGIML repo root.')

REPO_ROOT = find_repo_root()
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tools.infer_helpers import (
    _to_chw_rgb,
    _to_hw_mask,
    _parse_meta,
    _dataset_name,
    iter_prepared_samples,
    compute_binary_metrics,
    save_sample_plot
)

assert CHECKPOINT_PATH.exists(), f'Checkpoint not found: {CHECKPOINT_PATH}'
model, device, ckpt_info = load_model_from_checkpoint(CHECKPOINT_PATH)
normalization_mode = resolve_normalization_mode_for_inference(checkpoint_path=CHECKPOINT_PATH, default_mode='imagenet')
threshold_used = float(ckpt_info.get('default_threshold', 0.5) if THRESHOLD_FOR_METRICS is None else THRESHOLD_FOR_METRICS)
print('Device:', device)
print('Normalization:', normalization_mode)
print('Threshold for CSV metrics:', threshold_used)
print('Plot threshold:', PLOT_BINARY_THRESHOLD)

In [ ]:
npz_files = list(HF_SNAPSHOT_LOCAL_DIR.rglob('*.npz'))
tar_files = list(HF_SNAPSHOT_LOCAL_DIR.rglob('*.tar'))
total_samples = len(npz_files) + sum(
    len([m for m in tarfile.open(tar_path, mode='r').getmembers() if m.isfile() and m.name.lower().endswith('.npz')])
    for tar_path in tar_files
)

snapshot_path = Path(snapshot_download(repo_id=HF_DATASET_REPO_ID, repo_type='dataset', local_dir=str(HF_SNAPSHOT_LOCAL_DIR), local_dir_use_symlinks=False))
print('Snapshot:', snapshot_path)

rows = []
plot_samples: dict[str, list[dict]] = {}

for sample_uri, data in tqdm(iter_prepared_samples(snapshot_path), desc='Inference', total=total_samples):
    if 'image' not in data:
        continue

    image_chw = _to_chw_rgb(np.asarray(data['image']))
    h, w = int(image_chw.shape[1]), int(image_chw.shape[2])
    mask_hw = _to_hw_mask(data.get('mask'), h, w)
    meta = _parse_meta(data.get('metadata_json'))
    dataset = _dataset_name(sample_uri, meta)

    image_t = torch.from_numpy(image_chw).float()
    if image_t.max() > 1.0:
        image_t = image_t / 255.0

    prob = predict_probability_map_by_strategy(
        model=model,
        image=image_t,
        device=device,
        strategy=INFERENCE_STRATEGY,
        normalization_mode=normalization_mode,
    ).clamp(0.0, 1.0)

    prob_hw = prob.detach().cpu().numpy().astype(np.float32)
    pred_bin_metric = (prob_hw >= float(threshold_used)).astype(np.uint8)
    pred_bin_05 = (prob_hw >= float(PLOT_BINARY_THRESHOLD)).astype(np.uint8)
    m = compute_binary_metrics(pred_bin_metric, mask_hw)

    row = {
        'dataset': dataset, 'sample_uri': sample_uri, 'split': str(meta.get('split', 'test')),
        'label': int(meta.get('label', int(mask_hw.max() > 0))), 'strategy': INFERENCE_STRATEGY,
        'normalization_mode': normalization_mode, 'threshold_for_metrics': float(threshold_used),
        'plot_binary_threshold': float(PLOT_BINARY_THRESHOLD), 'height': h, 'width': w,
        'mean_probability': float(prob_hw.mean()), 'max_probability': float(prob_hw.max()),
        'pred_positive_ratio_threshold': float(pred_bin_metric.mean()),
        'pred_positive_ratio_0_5': float(pred_bin_05.mean()), 'gt_positive_ratio': float(mask_hw.mean())
    }
    row.update(m)
    rows.append(row)

    ds_bucket = plot_samples.setdefault(dataset, [])
    if len(ds_bucket) < 5:
        ds_bucket.append({'sample_uri': sample_uri, 'image_chw': image_chw, 'mask_hw': mask_hw, 'prob_hw': prob_hw, 'bin05_hw': pred_bin_05})

results_df = pd.DataFrame(rows).sort_values(['dataset', 'sample_uri']).reset_index(drop=True)
if results_df.empty:
    raise RuntimeError('No samples processed from HF snapshot.')

results_csv = CSV_OUTPUT_DIR / 'ngiml_hf_test_inference_results.csv'
summary_csv = CSV_OUTPUT_DIR / 'ngiml_hf_test_inference_summary_by_dataset.csv'
results_df.to_csv(results_csv, index=False)

summary_df = results_df.groupby('dataset', as_index=False).agg({
    'sample_uri': 'count', 'f1': 'mean', 'iou': 'mean', 'precision': 'mean', 'recall': 'mean',
    'accuracy': 'mean', 'mean_probability': 'mean', 'pred_positive_ratio_threshold': 'mean', 'gt_positive_ratio': 'mean'
}).rename(columns={'sample_uri': 'num_samples'})
summary_df.to_csv(summary_csv, index=False)

for ds_name, samples in sorted(plot_samples.items()):
    ds_dir = PLOT_OUTPUT_DIR / ds_name
    for i, sample in enumerate(samples, start=1):
        out_png = ds_dir / f'{ds_name}_sample_{i:02d}.png'
        title = f"{ds_name} | sample {i} | {Path(sample['sample_uri'].split('::')[0]).name}"
        save_sample_plot(out_png, sample['image_chw'], sample['mask_hw'], sample['prob_hw'], sample['bin05_hw'], title)

print('Saved full CSV:', results_csv)
print('Saved summary CSV:', summary_csv)
print('Saved plot root:', PLOT_OUTPUT_DIR)
display(summary_df)

In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

from tools.infer_helpers import load_model_from_checkpoint

try:
    from thop import clever_format, profile
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "thop"])
    from thop import clever_format, profile

# Load training configuration from the checkpoint
checkpoint = torch.load(CKPT_PATH, map_location="cpu")
training_config = checkpoint.get("training_config", {})

RUN_OUTPUT_DIR = Path(training_config.get("output_dir", OUTPUT_DIR))
CHECKPOINT_DIR = RUN_OUTPUT_DIR / "checkpoints"
if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError(f"Checkpoint directory not found: {CHECKPOINT_DIR}")

TARGET_EPOCH = None

if TARGET_EPOCH is None:
    best_ckpt = CHECKPOINT_DIR / "best_checkpoint.pt"
    if not best_ckpt.exists():
        raise FileNotFoundError("best_checkpoint.pt not found.")
    checkpoint_paths = [best_ckpt]
    CKPT_PATH = best_ckpt

else:
    target_ckpt = CHECKPOINT_DIR / f"checkpoint_epoch_{TARGET_EPOCH:03d}.pt"

    if target_ckpt.exists():
        checkpoint_paths = [target_ckpt]
        CKPT_PATH = target_ckpt
    else:
        checkpoint_paths = sorted(
            CHECKPOINT_DIR.glob(f"checkpoint_epoch_{TARGET_EPOCH}*.pt"),
            key=lambda p: p.stat().st_mtime
        )

        if not checkpoint_paths:
            raise FileNotFoundError(
                f"No checkpoint found for epoch {TARGET_EPOCH}."
            )
        CKPT_PATH = checkpoint_paths[-1]

model, _, ckpt_info = load_model_from_checkpoint(CKPT_PATH)
model = model.cpu().eval()

input_size = int(training_config.get("input_size", 448))

class _NgimlForProfiling(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        out = self.base_model(x, target_size=x.shape[-2:], residual_noise=None)
        if isinstance(out, (list, tuple)):
            return out[0]
        return out

wrapper = _NgimlForProfiling(model).eval()
dummy = torch.randn(1, 3, input_size, input_size, dtype=torch.float32)

with torch.no_grad():
    macs, params = profile(wrapper, inputs=(dummy,), verbose=False)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
flops = 2.0 * macs

macs_hr, params_hr = clever_format([macs, params], "%.3f")
flops_hr = clever_format([flops], "%.3f")

print("Checkpoint:", CKPT_PATH)
print("Target epoch:", TARGET_EPOCH)
print("Input shape:", tuple(dummy.shape))
print("Trainable params:", f"{trainable_params:,}")
print("Total params:", f"{total_params:,}")
print("THOP params:", params_hr)
print("MACs:", macs_hr)
print("Approx FLOPs (2 * MACs):", flops_hr)